In [6]:
from dotenv import load_dotenv
from langchain.tools import tool
from langchain_core.messages import HumanMessage
from typing import Dict, Any
from tavily import TavilyClient
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain_cohere import ChatCohere


load_dotenv()

True

In [2]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "travel_server" : {
            "transport" : "http",
            "url" : "https://mcp.kiwi.com"
        }
    }
)

tools = await client.get_tools()

In [4]:
model = ChatCohere(model="command-r-08-2024", temperature=0)
travel_agent = create_agent(model,
                     tools=tools,
                     checkpointer=InMemorySaver(),
                     system_prompt="You are a travel agent. Your job is to know the users needs and recommend flights according to that and No Follow UP Questions"
                     )

In [5]:

response = await travel_agent.ainvoke(
    {"messages" : [HumanMessage(content="Book me a flight from Patna to Delhi one passenger for 5 October")]},
    {"configurable" : {"thread_id" : "flight"}}
)

print(response["messages"][-1].content)


I've found a number of flights from Patna to Delhi on 26 September 2026. Here are the cheapest and shortest options:

## Cheapest
- Route: Patna → Delhi
- Times & duration: 22:10 → 23:45 (1h 35m)
- Cabin: Economy
- Price: 70 EUR
- Booking link: https://kiwi.com/u/wjr97q

## Shortest
- Route: Patna → Delhi
- Times & duration: 10:20 → 12:20 (2h)
- Cabin: Economy
- Price: 87 EUR
- Booking link: https://kiwi.com/u/b3wqcw

I recommend the shortest option, which is also the most expensive. Have a nice trip!


In [3]:
tavily_client = TavilyClient()

@tool
def web_search(query: str) -> str:
    """Search the web for information"""

    return tavily_client.search(query)

In [9]:
model = ChatCohere(model="command-r-08-2024", temperature=1.0)
venue_agent = create_agent(model,
                           tools=[web_search],
                           checkpointer=InMemorySaver(),
                           system_prompt="You are a venue Planner for the Wedding, Your Job is to Recommend the best Venue according to User's need")

In [12]:
response =  venue_agent.invoke({"messages" : [HumanMessage(content="I want to have the wedding in a place which is near sea")]},
                                    {"configurable" : {"thread_id" : "venue"}})

print(response["messages"][-1].content)

Certainly! Having a wedding near the sea is a wonderful idea. Here are some additional venue options to consider:

1. Oceanfront Hotel: Look for a hotel or resort that boasts an oceanfront location. These venues often have dedicated wedding coordinators and can offer a range of packages to suit your needs. You can have your ceremony on the beach, followed by a reception in a beautiful ballroom or outdoor terrace with breathtaking sea views.

2. Lighthouse Wedding: If you're seeking a unique and iconic setting, consider a lighthouse wedding. Many lighthouses offer event spaces with panoramic views of the ocean. The historic charm and dramatic backdrop of a lighthouse can create an unforgettable wedding experience.

3. Private Beach Club: Exclusive beach clubs often provide a luxurious and intimate setting for weddings. These venues typically have private beaches, elegant dining areas, and stunning sea vistas. A beach club wedding ensures a relaxed and sophisticated atmosphere for you an

In [7]:
model = ChatCohere(model="command-r-08-2024", temperature=1.0)
music_agent = create_agent(model,
                           tools=[web_search],
                           checkpointer=InMemorySaver(),
                           system_prompt="""You are a Wedding Music Curator Your job is to match music genres and styles according to what the couple and their guests like, Ask about their Favorite Artists, the vibe they want(e.g. Romantic, energetic, beachy) and Recommend Specific Genres""")

In [8]:
response = music_agent.invoke({"messages" : [HumanMessage(content="Well Romantic songs now you suggest the artists and songs")]},
                              {"configurable" : {"thread_id" : "music"}})

print(response["messages"][-1].content)

Sure! Here are some popular romantic songs and artists:

- "Someone You Loved" by Lewis Capaldi
- "How Deep Is Your Love" by the Bee Gees
- "The Power of Love" by Huey Lewis
- "The Greatest Love of All" by Whitney Houston
- "I Love You Always Forever" by Donna Lewis
- "Justify My Love" by Madonna
- "If It's Lovin' That You Want" by Rihanna
- "Hallelujah I Love Her So" by Ray Charles
- "Crazy for You" by Madonna
- "Cupid" by Sam Cooke
- "And I Love Her" by The Beatles
- "The Book of Love" by Magnetic Fields
- "I Will Always Love You" by Whitney Houston
- "Love Me Tender" by Elvis Presley
- "My Heart Will Go On" by Celine Dion
- "Open Arms" by Journey
- "Waiting for a Girl Like You" by Foreigner
- "Best of My Love" by The Emotions
- "Faithfully" by Journey
- "Three Times a Lady" by Commodores
- "Alone" by Heart


In [8]:
@tool
def music_specialist(query:str)-> str:
    """Delegate Music genres, Artists and specific Music to the Agent"""

    result = music_agent.invoke({"messages" : [HumanMessage(content=query)]},
                              {"configurable" : {"thread_id" : "music"}})

    return result["messages"][-1].content

Sure! Here are some popular romantic songs and artists:

- "Someone You Loved" by Lewis Capaldi
- "How Deep Is Your Love" by the Bee Gees
- "The Power of Love" by Huey Lewis
- "The Greatest Love of All" by Whitney Houston
- "I Love You Always Forever" by Donna Lewis
- "Justify My Love" by Madonna
- "If It's Lovin' That You Want" by Rihanna
- "Hallelujah I Love Her So" by Ray Charles
- "Crazy for You" by Madonna
- "Cupid" by Sam Cooke
- "And I Love Her" by The Beatles
- "The Book of Love" by Magnetic Fields
- "I Will Always Love You" by Whitney Houston
- "Love Me Tender" by Elvis Presley
- "My Heart Will Go On" by Celine Dion
- "Open Arms" by Journey
- "Waiting for a Girl Like You" by Foreigner
- "Best of My Love" by The Emotions
- "Faithfully" by Journey
- "Three Times a Lady" by Commodores
- "Alone" by Heart


In [12]:
@tool
def venue_specialist(query:str) -> str:
    """Delegate Venue Place, Pricing and Specifications of Venue to the Agent"""

    result = venue_agent.invoke({
        "messages" : [HumanMessage(content=query)]
    },
        {"configurable" : {"thread_id" : "venue"}})

    return result["messages"][-1].content

Certainly! Having a wedding near the sea is a wonderful idea. Here are some additional venue options to consider:

1. Oceanfront Hotel: Look for a hotel or resort that boasts an oceanfront location. These venues often have dedicated wedding coordinators and can offer a range of packages to suit your needs. You can have your ceremony on the beach, followed by a reception in a beautiful ballroom or outdoor terrace with breathtaking sea views.

2. Lighthouse Wedding: If you're seeking a unique and iconic setting, consider a lighthouse wedding. Many lighthouses offer event spaces with panoramic views of the ocean. The historic charm and dramatic backdrop of a lighthouse can create an unforgettable wedding experience.

3. Private Beach Club: Exclusive beach clubs often provide a luxurious and intimate setting for weddings. These venues typically have private beaches, elegant dining areas, and stunning sea vistas. A beach club wedding ensures a relaxed and sophisticated atmosphere for you an

In [ ]:
@tool
async def travel_specialist(query: str) -> str:
    """Delegate flight, pricing, and travel search requests to the travel specialist"""

    result = await travel_agent.ainvoke(
        {"messages" : [HumanMessage(content=query)]},
        {"configurable" : {"thread_id" : "isolated_travel_thread"}}
    )

    return result["messages"][-1].content
